# AI Recommendation System dengan Pretrained Model
**Studi kasus:** rekomendasi produk ckincare berdasarkan keluhan user

## 1. Gambaran alur sistem

1. **Input pengguna** — misalnya, “Kulit kering dan kusam”
2. **Katalog produk** — setiap item memiliki nama dan deskripsi.
3. **Pretrained embedding model** — mengubah teks menjadi vektor numerik.
4. **Cosine similarity** — membandingkan vektor preferensi dengan vektor produk.
5. **Ranking** — menampilkan produk dengan skor tertinggi.

**Istilah penting**
- **Pretrained model:** model yang bobotnya sudah dipelajari sebelumnya dari data besar.
- **Embedding:** representasi numerik dari teks.
- **Cosine similarity:** ukuran kemiripan arah dua vektor; nilai lebih tinggi berarti lebih mirip menurut representasi model.
- **Top-K:** sejumlah K rekomendasi teratas.

**Batasan:** contoh ini adalah *content-based recommendation*, bukan collaborative filtering. Sistem tidak mempelajari pola pembelian atau perilaku pengguna lain.

## 2. Instalasi library

In [ ]:
%pip -q install sentence-transformers scikit-learn pandas

## 3. Import library
`SentenceTransformer` untuk embedding, dan `cosine_similarity` untuk menghitung kemiripan.

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 4. Membuat dataset mini
Dataset diambil dari sample dataset pada `huggingface.co`

In [ ]:
ingredient = pd.read_csv("hf://datasets/Scanalyze/Skincare-ingredients/copy paste final.csv")
ingredient.head()

,Ingredient Name,Function,Safety Rating,Brief Description
0,Sodium Lauryl Sulfate (SLS),Surfactant,Moderate Risk,"Creates foam in cleansers, may cause irritation."
1,Glycerin,Humectant,Safe,"Retains moisture, keeps skin hydrated."
2,Benzyl Alcohol,Preservative,Moderate Risk,"Prevents bacteria, can be irritating in high a..."
3,Mineral Oil,Emollient,Safe,"Locks in moisture, but can feel heavy on the s..."
4,Mineral Oil,Emollient,Safe,"Locks in moisture, but can feel heavy on the s..."


## 5. Memuat pretrained model
`sentence-transformers/all-MiniLM-L6-v2`, model embedding yang relatif ringan dan menghasilkan vektor 384 dimensi.



In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)
print("Model berhasil dimuat:", MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model berhasil dimuat: sentence-transformers/all-MiniLM-L6-v2


## 6. Mengubah deskripsi produk menjadi embedding
Kita menghitung embedding produk satu kali, lalu menggunakannya berulang kali saat pengguna mengirim preferensi.

`normalize_embeddings=True` menormalkan vektor. Ini memudahkan perbandingan cosine similarity.

In [ ]:
product_embeddings = model.encode(
    ingredient["Brief Description"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Bentuk matriks embedding:", product_embeddings.shape)
print("Jumlah produk:", len(ingredient))
print("Dimensi embedding per produk:", product_embeddings.shape[1])

Batches:   0%|          | 0/34 [00:00<?, ?it/s]

Bentuk matriks embedding: (1060, 384)
Jumlah produk: 1060
Dimensi embedding per produk: 384


**Cara membaca output:** jika bentuknya `(6, 384)`, artinya ada 6 produk dan setiap produk direpresentasikan oleh 384 angka. Angka-angka ini bukan label produk; mereka merepresentasikan pola semantik yang dipelajari model.

## 7. Membuat fungsi rekomendasi
Fungsi berikut menerima preferensi dalam bentuk teks, membuat embedding untuk preferensi tersebut, menghitung similarity terhadap semua produk, lalu mengembalikan Top-K produk dengan skor tertinggi.

In [ ]:
def recommend_products(preference, top_k=3):
    # Validasi input sederhana
    if not isinstance(preference, str) or not preference.strip():
        raise ValueError("Preferensi harus berupa teks yang tidak kosong.")

    if top_k < 1:
        raise ValueError("top_k harus minimal 1.")

    # Embedding preferensi pengguna
    query_embedding = model.encode(
        [preference],
        normalize_embeddings=True
    )

    # Skor kemiripan terhadap seluruh produk
    scores = cosine_similarity(
        query_embedding,
        product_embeddings
    )[0]

    # Ambil indeks dengan skor tertinggi
    ranked_indices = scores.argsort()[::-1][:min(top_k, len(ingredient))]

    results = ingredient.iloc[ranked_indices].copy()
    results["skor_kemiripan"] = scores[ranked_indices].round(3)

    return results.reset_index(drop=True)

### Memahami kode
- `model.encode([preference])`: mengubah input pengguna menjadi embedding.
- `cosine_similarity(...)`: menghasilkan skor untuk setiap produk.
- `argsort()[::-1]`: mengurutkan indeks dari skor tertinggi ke terendah.
- `top_k`: membatasi jumlah rekomendasi yang ditampilkan.


## 8. Uji rekomendasi pertama

In [ ]:
user_preference = (
    "recomendation for PIH acne scar"
)

recommend_products(user_preference, top_k=3)

,Ingredient Name,Function,Safety Rating,Brief Description,skor_kemiripan
0,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.491
1,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.491
2,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.491


## 9. Eksperimen: ubah preferensi pengguna

In [ ]:
test_preferences = [
    "Minimizing facial pores",
    "Reducing oily skin and red, inflamed acne",
    "remove dark spots from acne scars",
    "How to keep the skin hydrated"
]

for preference in test_preferences:
    print("\nPreferensi:", preference)
    display(recommend_products(preference, top_k=5))


Preferensi: Minimizing facial pores


,Ingredient Name,Function,Safety Rating,Brief Description,skor_kemiripan
0,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.583
1,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.583
2,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.583
3,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.583
4,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.583



Preferensi: Reducing oily skin and red, inflamed acne


,Ingredient Name,Function,Safety Rating,Brief Description,skor_kemiripan
0,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.643
1,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.643
2,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.643
3,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.643
4,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.643



Preferensi: remove dark spots from acne scars


,Ingredient Name,Function,Safety Rating,Brief Description,skor_kemiripan
0,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.524
1,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.524
2,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.524
3,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.524
4,Salicylic Acid,Exfoliant,Safe,Unclogs pores and treats acne.,0.524



Preferensi: How to keep the skin hydrated


,Ingredient Name,Function,Safety Rating,Brief Description,skor_kemiripan
0,Saccharide Isomerate,Humectant,Safe,Binds water to the skin for long-lasting hydra...,0.782
1,Cucumber Extract,Cooling,Safe,Soothes and hydrates skin.,0.729
2,Rose Water,Hydrating,Safe,"Refreshes skin, provides light hydration.",0.719
3,Glycerin,Humectant,Safe,"Retains moisture, keeps skin hydrated.",0.702
4,Glycerin,Humectant,Safe,"Retains moisture, keeps skin hydrated.",0.702


## 10. Tampilkan semua skor

In [ ]:
def score_all_products(preference):
    query_embedding = model.encode(
        [preference],
        normalize_embeddings=True
    )
    scores = cosine_similarity(query_embedding, product_embeddings)[0]

    result = ingredient.copy()
    result["skor_kemiripan"] = scores.round(3)
    return result.sort_values("skor_kemiripan", ascending=False).reset_index(drop=True)

score_all_products("How to hydrate skin")

,Ingredient Name,Function,Safety Rating,Brief Description,skor_kemiripan
0,Lactic Acid,Exfoliant,Safe,Gently exfoliates and hydrates skin.,0.756
1,Lactic Acid,Exfoliant,Safe,Gently exfoliates and hydrates skin.,0.756
2,Lactic Acid,Exfoliant,Safe,Gently exfoliates and hydrates skin.,0.756
3,Lactic Acid,Exfoliant,Safe,Gently exfoliates and hydrates skin.,0.756
4,Lactic Acid,Exfoliant,Safe,Gently exfoliates and hydrates skin.,0.756
...,...,...,...,...,...
1055,Parabens,Preservative,Moderate Risk,"Prevents microbial growth, but some concerns o...",0.002
1056,Parabens,Preservative,Moderate Risk,"Prevents microbial growth, but some concerns o...",0.002
1057,Parabens,Preservative,Moderate Risk,"Prevents microbial growth, but some concerns o...",0.002
1058,Parabens,Preservative,Moderate Risk,"Prevents microbial growth, but some concerns o...",0.002
